In [1]:
import ROOT
import os

ROOT.gROOT.SetBatch(True)
ROOT.gErrorIgnoreLevel = ROOT.kError
ROOT.gStyle.SetOptStat(0)  # 통계창 끄기 (원하면 주석 처리)

# ─── 0) 입력 파일 & 출력 디렉토리 ──────────────────────────────────────
# (ROOT 파일 이름, 라벨)
input_samples = [
    ("hist_ST_s2018.root",  "ST_s_2018"),
    ("hist_ST_s2024.root",  "ST_s_2024"),
    ("hist_ST_t2018.root",  "ST_t_2018"),
    ("hist_ST_t2024.root",  "ST_t_2024"),
    ("hist_ST_tW2018.root", "ST_tW_2018"),
    ("hist_ST_tW2024.root", "ST_tW_2024"),
    ("hist_TT2018.root",    "TT_2018"),
    ("hist_TT2024.root",    "TT_2024"),
]

out_dir = "2D_compare_ST_like_overlay_norm_2D"
os.makedirs(out_dir, exist_ok=True)

# 2D 그림 저장 폴더
plot2d_dir = os.path.join(out_dir, "2D_plots_colz")
os.makedirs(plot2d_dir, exist_ok=True)

# ─── 1) 'plots' 디렉토리 안에서 2D 히스토그램 이름만 가져오기 ────────
def get_hist2d_names_from_root(root_file):
    f = ROOT.TFile.Open(root_file)
    if not f or f.IsZombie():
        raise RuntimeError(f"Cannot open file: {root_file}")
    plots_dir = f.Get("plots")
    if not plots_dir:
        raise RuntimeError(f"'plots' directory not found in {root_file}")

    names = []
    for key in plots_dir.GetListOfKeys():
        obj = key.ReadObj()
        # TH2 계열이면서 dimension=2 인 것만 (TH2F, TH2D, TH2I 등 모두 포함)
        if obj.InheritsFrom("TH2") and obj.GetDimension() == 2:
            names.append(obj.GetName())

    f.Close()
    return names

# 첫 번째 파일 기준으로 2D 히스토그램 이름 리스트 얻기
hist2d_names = get_hist2d_names_from_root(input_samples[0][0])
print(f"Found {len(hist2d_names)} 2D histograms in 'plots/' of {input_samples[0][0]}:")
for n in hist2d_names:
    print("  ", n)

# ─── 2) 각 파일에서 해당 2D 히스토그램을 COLZ로 그림 & 저장 ────────
canvas = ROOT.TCanvas("c2d", "c2d", 800, 700)

for root_file, label in input_samples:
    print(f"\n[INFO] Processing file: {root_file} (label={label})")

    if not os.path.isfile(root_file):
        print(f"  [WARNING] File not found: {root_file}, skip.")
        continue

    f = ROOT.TFile.Open(root_file)
    if not f or f.IsZombie():
        print(f"  [WARNING] Cannot open file: {root_file}, skip.")
        continue

    plots_dir = f.Get("plots")
    if not plots_dir:
        print(f"  [WARNING] 'plots' directory not found in {root_file}, skip.")
        f.Close()
        continue

    file_base = os.path.splitext(os.path.basename(root_file))[0]

    for hname in hist2d_names:
        hobj = plots_dir.Get(hname)
        if not hobj:
            print(f"    [WARNING] Histogram '{hname}' not found in {root_file}, skip.")
            continue
        if not (hobj.InheritsFrom("TH2") and hobj.GetDimension() == 2):
            print(f"    [WARNING] Object '{hname}' is not 2D TH2 in {root_file}, skip.")
            continue

        hist2d = hobj

        canvas.cd()
        canvas.Clear()

        hist2d.SetTitle(f"{label} : {hname}")  # 라벨에 year 포함

        hist2d.Draw("COLZ")

        safe_hname = hname.replace("/", "_")
        #out_name = f"{file_base}__{safe_hname}.png"
        out_name = f"{safe_hname}_{file_base}.png"
        out_path = os.path.join(plot2d_dir, out_name)

        canvas.SaveAs(out_path)
        print(f"    [SAVED] {out_path}")

    f.Close()

print("\n[DONE] All 2D COLZ plots have been created in:", plot2d_dir)


Found 16 2D histograms in 'plots/' of hist_ST_s2018.root:
   top_pt_vs_eta
   b_pt_vs_eta
   bb_pt_vs_eta
   lep_pt_vs_eta
   b_eta_vs_lep_eta
   b_eta_vs_bb_eta
   bb_eta_vs_lep_eta
   b_pt_vs_lep_pt
   bb_pt_vs_lep_pt
   bb_pt_vs_b_pt
   t_pt_vs_b_pt
   t_pt_vs_lep_pt
   t_pt_vs_bb_pt
   t_eta_vs_lep_eta
   t_eta_vs_b_eta
   t_eta_vs_bb_eta

[INFO] Processing file: hist_ST_s2018.root (label=ST_s_2018)
    [SAVED] 2D_compare_ST_like_overlay_norm_2D/2D_plots_colz/top_pt_vs_eta_hist_ST_s2018.png
    [SAVED] 2D_compare_ST_like_overlay_norm_2D/2D_plots_colz/b_pt_vs_eta_hist_ST_s2018.png
    [SAVED] 2D_compare_ST_like_overlay_norm_2D/2D_plots_colz/bb_pt_vs_eta_hist_ST_s2018.png
    [SAVED] 2D_compare_ST_like_overlay_norm_2D/2D_plots_colz/lep_pt_vs_eta_hist_ST_s2018.png
    [SAVED] 2D_compare_ST_like_overlay_norm_2D/2D_plots_colz/b_eta_vs_lep_eta_hist_ST_s2018.png
    [SAVED] 2D_compare_ST_like_overlay_norm_2D/2D_plots_colz/b_eta_vs_bb_eta_hist_ST_s2018.png
    [SAVED] 2D_compare_ST_like_ov